# YOLOv11 Retraining & ONNX Export Pipeline

This notebook downloads custom Roboflow dataset (`smart_cooling_edge_ai`), fine-tunes the **YOLOv11** student detection model for 100 epochs, evaluates performance, and automatically exports the final model to **`models/yolo11n.onnx`**.

### Step 1: Install Dependencies & Download Dataset from Roboflow

In [ ]:
!pip install -q ultralytics roboflow onnx onnxruntime

from roboflow import Roboflow
rf = Roboflow(api_key="your-roboflow-apikey")
project = rf.workspace("your-workspace").project("your_project_name")
dataset = project.version(1).download("yolov11")

loading Roboflow workspace...
loading Roboflow project...
Exporting format yolov11 in progress : 0.0%
Version export complete for yolov11 format



Extracting Dataset Version Zip to smart_cooling_edge_ai-1 in yolov11:: 100%|██████████| 381/381 [00:00<00:00, 1920.99it/s]


### Step 2: Train YOLOv11n Model

Before training,

* Ensure you have the Installed necessary dependencies, including `ultralytics` and `roboflow`.
* Ensure your PC has a GPU with CUDA support for faster training. If not, you can still train on CPU, but it will be significantly slower. And also, make sure you have enough RAM and disk space to handle the dataset and model files. 
* Ensure you have the necessary permissions to access the Roboflow dataset. You may need to sign up for a Roboflow account and obtain an API key to download the dataset.
* Ensure you have pytorch installed with the correct version that is compatible with your GPU and CUDA version. You can check the compatibility matrix on the PyTorch website.
* Othrerwise, you can use Google Colab or Kaggle for free GPU access.

In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"


In [2]:
from ultralytics import YOLO
import torch

In [ ]:
# Check is GPU available for training.
device = 0 if torch.cuda.is_available() else 'cpu'
print(f"Training device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

Training device: 0 (NVIDIA GeForce GTX 1650)


In [ ]:
# Locate data.yaml inside downloaded dataset
dataset_yaml = "./smart_cooling_edge_ai-1/data.yaml"
print(f"Dataset location: {dataset_yaml}")

# Load base YOLOv11 nano model pretrained weights
model = YOLO("yolo11n.pt")

# Fine-tune YOLOv11n model on GPU
results = model.train(
    data=dataset_yaml,
    epochs=300,
    imgsz=640,
    batch=16,
    workers=4,
    device=device,
    name="yolo11n_retrained"
)


Dataset location: ./smart_cooling_edge_ai-1/data.yaml
Ultralytics 8.4.126  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./smart_cooling_edge_ai-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum

### Step 3: Evaluate Model Accuracy (mAP50)

In [5]:
# Validate model on validation dataset split
metrics = model.val()
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"mAP50:    {metrics.box.map50:.4f}")

Ultralytics 8.4.126  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce GTX 1650, 4096MiB)
YOLO11n summary (fused): 101 layers, 2,582,347 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 249.9156.2 MB/s, size: 99.4 KB)
val: Scanning D:\DEPARTMENT OF COMPUTER SCIENCE\Computer Science\AI ML\AI_ML_DL\Computer Vision Projects\Smart-cooling-edge_AI\notebooks\smart_cooling_edge_ai-1\valid\labels.cache... 37 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 37/37 32.9Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 2.6s/it 7.7s2.3s2s
                   all         37        337      0.898      0.837      0.875      0.362
Speed: 10.3ms preprocess, 26.2ms inference, 0.0ms loss, 3.8ms postprocess per image
Results saved to D:\DEPARTMENT OF COMPUTER SCIENCE\Computer Science\AI ML\AI_ML_DL\Computer Vision Projects\Smart-cooling-edge_AI\runs\detect\val
mAP50-95: 0.3618
mAP50:    0.875

### Step 4: Export Retrained Model to ONNX & Replace System Model

In [3]:
import os
import shutil
from ultralytics import YOLO

# Load the fine-tuned model directly from disk
best_weights_path = "../runs/detect/yolo11n_retrained/weights/best.pt"
print(f"[i] Loading trained weights from: {best_weights_path}")
model = YOLO(best_weights_path)

# Export trained weights to ONNX format (640x640 resolution)
onnx_file_path = model.export(format="onnx", imgsz=640, simplify=True)
print(f"[✔] Exported ONNX model to: {onnx_file_path}")

# Deploy exported ONNX file to production path (models/yolo11n.onnx)
target_onnx_path = "../models/yolo11n.onnx"
os.makedirs(os.path.dirname(target_onnx_path), exist_ok=True)
shutil.copy(onnx_file_path, target_onnx_path)
print(f"[✔] Successfully updated production ONNX model at: {target_onnx_path}")


[i] Loading trained weights from: ../runs/detect/yolo11n_retrained/weights/best.pt
Ultralytics 8.4.115  Python-3.10.20 torch-2.13.0+cpu CPU (12th Gen Intel Core i5-12450H)
YOLO11n summary (fused): 101 layers, 2,582,347 parameters, 0 gradients, 6.4 GFLOPs

PyTorch: starting from '..\runs\detect\yolo11n_retrained\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (5.2 MB)

ONNX: starting export with onnx 1.22.0 opset 20...
ONNX: slimming with onnxslim 0.1.95...
ONNX: export success  5.7s, saved as '..\runs\detect\yolo11n_retrained\weights\best.onnx' (10.1 MB)

Export complete (6.1s)
Results saved to D:\DEPARTMENT OF COMPUTER SCIENCE\Computer Science\AI ML\AI_ML_DL\Computer Vision Projects\Smart-cooling-edge_AI\runs\detect\yolo11n_retrained\weights\best.onnx
Predict:         yolo predict task=detect model=..\runs\detect\yolo11n_retrained\weights\best.onnx imgsz=640 
Validate:        yolo val task=detect model=..\runs\detect\yolo11n_retrained\weights\b